In [1]:
# ── Import Libraries ─────────────────────────────────
import pandas as pd
import numpy as np
import warnings

In [2]:
# ── Load all three sheets from the workbook ──────────
file_path = "../data/ecommerce_operations_data.xlsx"

orders    = pd.read_excel(file_path, sheet_name="orders")
customers = pd.read_excel(file_path, sheet_name="customers")
marketing = pd.read_excel(file_path, sheet_name="marketing_spend")

print(f"orders          : {orders.shape[0]} rows × {orders.shape[1]} columns")
print(f"customers       : {customers.shape[0]} rows × {customers.shape[1]} columns")
print(f"marketing_spend : {marketing.shape[0]} rows × {marketing.shape[1]} columns")

orders          : 150 rows × 11 columns
customers       : 40 rows × 5 columns
marketing_spend : 28 rows × 4 columns


In [3]:
# ── Schema check: data types ─────────────────────────
print("── orders ──")
print(orders.dtypes)
print("\n── customers ──")
print(customers.dtypes)
print("\n── marketing_spend ──")
print(marketing.dtypes)

── orders ──
order_id              int64
customer_id          object
order_date           object
region               object
channel              object
campaign             object
product_category     object
units_sold            int64
unit_price          float64
discount              int64
returned             object
dtype: object

── customers ──
customer_id     object
signup_date     object
segment         object
age            float64
city            object
dtype: object

── marketing_spend ──
week_start    object
channel       object
campaign      object
spend          int64
dtype: object


In [4]:
# ── Schema check: missingness rates (%) ──────────────
print("── orders ──")
print((orders.isnull().sum() / len(orders) * 100).round(2))
print("\n── customers ──")
print((customers.isnull().sum() / len(customers) * 100).round(2))
print("\n── marketing_spend ──")
print((marketing.isnull().sum() / len(marketing) * 100).round(2))

── orders ──
order_id             0.0
customer_id          0.0
order_date           0.0
region               0.0
channel              0.0
campaign            40.0
product_category     0.0
units_sold           0.0
unit_price           0.0
discount             0.0
returned             0.0
dtype: float64

── customers ──
customer_id     0.0
signup_date     0.0
segment         0.0
age            15.0
city            0.0
dtype: float64

── marketing_spend ──
week_start    0.0
channel       0.0
campaign      0.0
spend         0.0
dtype: float64


In [5]:
# ── Schema check: uniqueness for key columns ────────
oid_total  = len(orders)
oid_unique = orders["order_id"].nunique()
print(f"orders.order_id   : {oid_unique}/{oid_total} unique  →  "
      f"{'PASS' if oid_unique == oid_total else 'FAIL — duplicates detected'}")

cid_total  = len(customers)
cid_unique = customers["customer_id"].nunique()
print(f"customers.cust_id : {cid_unique}/{cid_total} unique  →  "
      f"{'PASS' if cid_unique == cid_total else 'FAIL — duplicates detected'}")

orphans = set(orders["customer_id"]) - set(customers["customer_id"])
print(f"Orphan customers (in orders but not in customers): "
      f"{orphans if orphans else 'None — PASS'}")

idle = set(customers["customer_id"]) - set(orders["customer_id"])
print(f"Idle customers   (in customers but no orders)    : "
      f"{idle if idle else 'None'}")

orders.order_id   : 150/150 unique  →  PASS
customers.cust_id : 40/40 unique  →  PASS
Orphan customers (in orders but not in customers): None — PASS
Idle customers   (in customers but no orders)    : {'C015'}


In [6]:
# ── Standardize categorical fields ───────────────────
str_cols_orders = ["customer_id", "region", "channel", "campaign",
                   "product_category", "returned"]
for col in str_cols_orders:
    orders[col] = orders[col].astype(str).str.strip().str.title()

orders["returned"] = orders["returned"].str.upper()
orders["campaign"] = orders["campaign"].replace("Nan", np.nan)

str_cols_customers = ["customer_id", "segment", "city"]
for col in str_cols_customers:
    customers[col] = customers[col].astype(str).str.strip().str.title()

str_cols_mktg = ["channel", "campaign"]
for col in str_cols_mktg:
    marketing[col] = marketing[col].astype(str).str.strip().str.title()

print("Standardization complete.")

Standardization complete.


In [7]:
# ── Standardize categorical fields ───────────────────
str_cols_orders = ["customer_id", "region", "channel", "campaign",
                   "product_category", "returned"]
for col in str_cols_orders:
    orders[col] = orders[col].astype(str).str.strip().str.title()

orders["returned"] = orders["returned"].str.upper()
orders["campaign"] = orders["campaign"].replace("Nan", np.nan)

str_cols_customers = ["customer_id", "segment", "city"]
for col in str_cols_customers:
    customers[col] = customers[col].astype(str).str.strip().str.title()

str_cols_mktg = ["channel", "campaign"]
for col in str_cols_mktg:
    marketing[col] = marketing[col].astype(str).str.strip().str.title()

print("Standardization complete.")

Standardization complete.


In [8]:
# ── Fill missing campaign for Organic orders ─────────
orders["campaign"] = orders["campaign"].fillna("No_Campaign")

print(f"Null campaigns remaining: {orders['campaign'].isnull().sum()}")

Null campaigns remaining: 0


In [9]:
# ── Derived metrics ──────────────────────────────────
orders["gross_revenue"] = orders["units_sold"] * orders["unit_price"]

orders["net_revenue"] = orders["gross_revenue"] * (1 - orders["discount"] / 100)

# net_revenue_after_returns: set to 0 when returned = 'Y'
orders["net_revenue_after_returns"] = np.where(
    orders["returned"] == "Y", 0, orders["net_revenue"]
)

# Assumption: COGS is roughly 55% of gross revenue (industry average estimate)
orders["estimated_cogs"] = orders["gross_revenue"] * 0.55
orders["margin"] = orders["net_revenue_after_returns"] - orders["estimated_cogs"]

print("Derived metrics created.")
print(orders[["order_id", "gross_revenue", "net_revenue",
              "net_revenue_after_returns", "estimated_cogs", "margin"]].head(5))

Derived metrics created.
   order_id  gross_revenue  net_revenue  net_revenue_after_returns  \
0      2001         119.94       59.970                       0.00   
1      2002         240.00      240.000                     240.00   
2      2003        1499.97     1499.970                    1499.97   
3      2004          19.99       15.992                       0.00   
4      2005          99.95       79.960                       0.00   

   estimated_cogs    margin  
0         65.9670  -65.9670  
1        132.0000  108.0000  
2        824.9835  674.9865  
3         10.9945  -10.9945  
4         54.9725  -54.9725  


In [11]:
# ── Convert dates and create week_start ──────────────
orders["order_date"]     = pd.to_datetime(orders["order_date"])
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
marketing["week_start"]  = pd.to_datetime(marketing["week_start"])

orders["week_start"] = orders["order_date"].dt.to_period("W-SUN").apply(
    lambda p: p.start_time
)

print("Dates converted and week_start created.")

Dates converted and week_start created.


In [12]:
# ── Aggregate weekly KPIs ────────────────────────────
weekly_kpis = orders.groupby(
    ["week_start", "channel", "campaign", "region", "product_category"]
).agg(
    total_orders       = ("order_id", "nunique"),
    unique_customers   = ("customer_id", "nunique"),
    gross_revenue      = ("gross_revenue", "sum"),
    net_revenue        = ("net_revenue", "sum"),
    net_rev_after_ret  = ("net_revenue_after_returns", "sum"),
    margin             = ("margin", "sum"),
    returned_count     = ("returned", lambda x: (x == "Y").sum()),
    total_units        = ("units_sold", "sum"),
).reset_index()

weekly_kpis["return_rate"] = (
    weekly_kpis["returned_count"] / weekly_kpis["total_orders"]
).round(4)

weekly_kpis["aov"] = (
    weekly_kpis["net_rev_after_ret"] / weekly_kpis["total_orders"]
).round(2)

print(f"weekly_kpis shape: {weekly_kpis.shape}")

weekly_kpis shape: (145, 15)


In [13]:
# ── Join marketing_spend and compute ROAS ────────────
roi = weekly_kpis.merge(
    marketing,
    on=["week_start", "channel", "campaign"],
    how="left"
)

roi["spend"] = roi["spend"].fillna(0)

roi["roas"] = np.where(
    roi["spend"] > 0,
    (roi["net_rev_after_ret"] / roi["spend"]).round(4),
    np.nan
)

roi["profit_after_spend"] = roi["margin"] - roi["spend"]

print(f"roi shape: {roi.shape}")

roi shape: (145, 18)


In [14]:
# ── Validation: distribution check & join cardinality ─
print("── Revenue distribution (net_rev_after_ret) ──")
print(weekly_kpis["net_rev_after_ret"].describe().round(2))

print("\n── AOV distribution ──")
print(weekly_kpis["aov"].describe().round(2))

print("\n── Join cardinality check ──")
print(f"weekly_kpis rows before join : {len(weekly_kpis)}")
print(f"roi rows after join          : {len(roi)}")
print(f"Row count match              : "
      f"{'PASS — no row duplication' if len(roi) == len(weekly_kpis) else 'FAIL — check for duplicates'}")

print(f"\n── Logical check ──")
total_rev_orders = orders["net_revenue_after_returns"].sum().round(2)
total_rev_kpis   = weekly_kpis["net_rev_after_ret"].sum().round(2)
print(f"Total net_rev_after_ret (orders) : {total_rev_orders}")
print(f"Total net_rev_after_ret (kpis)   : {total_rev_kpis}")
print(f"Match : {'PASS' if total_rev_orders == total_rev_kpis else 'FAIL'}")

── Revenue distribution (net_rev_after_ret) ──
count     145.00
mean      367.08
std       601.61
min         0.00
25%        39.98
50%       125.00
75%       311.50
max      2999.94
Name: net_rev_after_ret, dtype: float64

── AOV distribution ──
count     145.00
mean      363.16
std       602.13
min         0.00
25%        39.98
50%       120.00
75%       299.97
max      2999.94
Name: aov, dtype: float64

── Join cardinality check ──
weekly_kpis rows before join : 145
roi rows after join          : 145
Row count match              : PASS — no row duplication

── Logical check ──
Total net_rev_after_ret (orders) : 53226.05
Total net_rev_after_ret (kpis)   : 53226.05
Match : PASS


In [15]:
# ── Export CSVs ──────────────────────────────────────
weekly_kpis.to_csv("weekly_kpis.csv", index=False)
roi.to_csv("roi.csv", index=False)

print("weekly_kpis.csv exported — rows:", len(weekly_kpis))
print("roi.csv exported         — rows:", len(roi))

weekly_kpis.csv exported — rows: 145
roi.csv exported         — rows: 145
